# Step 1. Congruency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/1_congruency.ipynb)

A backup for the tutorial: the same step without an assistant. Run the cells from top to bottom.

In [ ]:
%pip install -q sweetpea pandas

**The question.** Does an irrelevant word interfere with naming a colour?

**Prediction.** Naming the ink is slower when the word names another colour (incongruent) than when it names the same one (congruent).

**The analysis.** Response time on congruent trials against incongruent trials.

**The decision.** With three colours there are 3 congruent and 6 incongruent pairs of ink and word. Two wishes pull apart: *every pair equally often* (then one third of the trials is congruent) or *half of the trials congruent* (then the word tells something about the ink). Below are both.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
# a derived factor: it follows from colour and word on the same trial
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


congruency = Factor("congruency", [
    DerivedLevel("congruent", WithinTrial(same, [colour, word])),
    DerivedLevel("incongruent", WithinTrial(different, [colour, word])),
])

### Option A: every pair equally often

Colour and word are crossed: each of the 9 pairs occurs equally often. `MinimumTrials(18)` asks for two rounds.

In [ ]:
block_a = CrossBlock(design=[colour, word, congruency], crossing=[colour, word],
                     constraints=[MinimumTrials(18)])
trials_a = pd.DataFrame(synthesize_trials(block_a, 1, CMSGen)[0])
print(pd.crosstab(trials_a["colour"], trials_a["word"]))
print(trials_a["congruency"].value_counts())

### Option B: half of the trials congruent

Every pair once, and a second copy of the congruent pairs only. `copy` is a helper factor; the unwanted second copies are excluded.

In [ ]:
copy = Factor("copy", ["first", "second"])


def unwanted(copy, colour, word):       # a second copy of an incongruent pair
    return copy == "second" and colour != word


def wanted(copy, colour, word):
    return not unwanted(copy, colour, word)


second_copy = Factor("second_copy", [
    DerivedLevel("unwanted", WithinTrial(unwanted, [copy, colour, word])),
    DerivedLevel("wanted", WithinTrial(wanted, [copy, colour, word])),
])
block_b = CrossBlock(design=[colour, word, congruency, copy, second_copy],
                     crossing=[colour, word, copy],
                     constraints=[Exclude((second_copy, "unwanted")), MinimumTrials(24)],
                     require_complete_crossing=False)
trials_b = pd.DataFrame(synthesize_trials(block_b, 1, CMSGen)[0])
print(pd.crosstab(trials_b["colour"], trials_b["word"]))
print(trials_b["congruency"].value_counts())

### What does each option cost?

How often is a word shown in its own colour? One third means the word tells nothing about the ink.

In [ ]:
for name, trials in [("every pair equally often", trials_a), ("half congruent", trials_b)]:
    print(name, "-> a word is in its own colour on", round((trials["word"] == trials["colour"]).mean() * 100), "% of trials")

**Check.** In which option could a participant guess the ink from the word? In which are there fewer congruent trials? Neither is free: which cost is acceptable depends on what you compare.

**Try.** Add a fourth colour to `colours` and run again. What happens to the share of congruent trials in option A?